In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

In [62]:
df = pd.read_csv('data/pima-indians-diabetes.data', skiprows=2, header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
y = np.where(y == 0, -1, 1).reshape(-1, 1)   # tanh output uses -1 / +1


In [63]:
def tanh(x):
    x = np.clip(x, -500, 500)
    return np.tanh(x)

def tanh_derivative(output):
    return 1 - output**2


hidden size: number of neurons in the hidden layer
patience: how long to wait to stop before accuracy is not improving. 
min_improvement: smallest imrpovememt required to consider is better
lambda_reg: regularization strength

Formular
Eaug(w) = Ein(w) + (λ / N) wT . w
where 
    Ein is insample error - Error on the training data. this the code we'll use it as MSE error (mean sqaured) 
    Eaug is augmented error - Training error + penalty for large weights. Adds regularization


In [64]:
def train_nn(
    X_train,
    y_train,
    X_val,          # <-- add this
    y_val,          # <-- add this
    lr=0.05,
    hidden_size=5,
    max_epochs=500,
    patience=50,
    min_improvement=1e-4,
    lambda_reg=0.001,
    seed=42,
    verbose=False
):
    np.random.seed(seed)

    n_samples, input_size = X_train.shape
    output_size = 1 # output neuron is size 1

    W1 = np.random.randn(input_size, hidden_size) * 0.01 # helps training be stable
    W2 = np.random.randn(hidden_size + 1, output_size) * 0.01

    best_val_loss = float("inf")
    best_W1 = W1.copy() # store copies of the best weights
    best_W2 = W2.copy() 
    epochs_without_improvement = 0 # count how many epochs have passed without improvment

    loss_history = []
    acc_history = []

    for epoch in range(max_epochs):

        # forward prop
        hidden_linear = X_train @ W1
        hidden_output = tanh(hidden_linear)

        hidden_output_bias = np.hstack((
            np.ones((hidden_output.shape[0], 1)),
            hidden_output
        ))

        output_linear = hidden_output_bias @ W2
        predicted_output = tanh(output_linear)

        # end forward prop

        # loss with L2 regularization
        mse_loss = np.mean((predicted_output - y_train) ** 2) # Ein(w) -> in sample error. 

        reg_W1 = lambda_reg * W1.copy()
        reg_W2 = lambda_reg * W2.copy()

        # do not regularize bias row
        reg_W1[0, :] = 0
        reg_W2[0, :] = 0

        # augmented error
        '''
        Eaug(w) = Ein(w) + (λ / N) wT . w
        
        res loss is  (λ / N) wT . w
        '''
        reg_loss = (lambda_reg / n_samples) * (
            np.sum(W1[1:, :] ** 2) + np.sum(W2[1:, :] ** 2)
        ) 

        total_loss = mse_loss + reg_loss # this is augmented error Eaug(w)

        # accuracy
        '''
        predicted_labels = [1, -1, 1]
        y_train          = [1,  1, 1]

        => [True, False, True]
        convert T to 1, F to 0
        => [1, 0, 1]
        '''
        predicted_labels = np.where(predicted_output >= 0, 1, -1) # convert continious output into class labels
        acc = np.mean(predicted_labels == y_train) # compute the % of accurate predictions

        # append to history for tracking
        loss_history.append(total_loss)
        acc_history.append(acc)

        # early stopping
        '''
        We are predicting the validation
        '''
        hidden_val = tanh(X_val @ W1)
        hidden_val_bias = np.hstack((
            np.ones((hidden_val.shape[0], 1)),
            hidden_val
        ))

        output_val = tanh(hidden_val_bias @ W2)
        val_loss = np.mean((output_val - y_val) ** 2) # MSE of the loss. 

        # backward pass
        delta_output = (predicted_output - y_train) * tanh_derivative(predicted_output)

        delta_hidden_full = (delta_output @ W2.T) * tanh_derivative(hidden_output_bias)
        delta_hidden = delta_hidden_full[:, 1:]   # remove bias column

        # batch gradients
        grad_W2 = (hidden_output_bias.T @ delta_output) / n_samples
        grad_W1 = (X_train.T @ delta_hidden) / n_samples

        # add regularization gradients
        grad_W2 += reg_W2
        grad_W1 += reg_W1

        # update
        W2 -= lr * grad_W2
        W1 -= lr * grad_W1

        # early stopping on validation loss
        if val_loss < best_val_loss - min_improvement:
            best_val_loss = val_loss
            best_W1 = W1.copy()
            best_W2 = W2.copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose and ((epoch + 1) % 50 == 0 or epoch == 0):
            print(f"Epoch {epoch+1}: Loss = {total_loss:.6f}, Accuracy = {acc:.4f}, Val Loss = {val_loss:.6f}")

        if epochs_without_improvement >= patience:
            break

    return best_W1, best_W2, loss_history, acc_history

In [65]:
def evaluate_nn(X_data, y_data, W1, W2):
    hidden_linear = X_data @ W1
    hidden_output = tanh(hidden_linear)

    hidden_output_bias = np.hstack((
        np.ones((hidden_output.shape[0], 1)),
        hidden_output
    ))

    output_linear = hidden_output_bias @ W2
    predicted_output = tanh(output_linear)

    predicted_labels = np.where(predicted_output >= 0, 1, -1)
    accuracy = np.mean(predicted_labels == y_data)

    return accuracy


In [66]:
kf = KFold(n_splits=10, shuffle=True, random_state=42) # split data into 10 parts

fold_accuracies = []

for fold_no, (train_index, val_index) in enumerate(kf.split(X), start=1):
    '''
    X_train_raw = training inputs
    y_train = training labels

    X_val_raw = validation inputs
    y_val = validation labels
    '''
    X_train_raw, X_val_raw = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    # fit scaler only on training fold. Normalization
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)

    # add bias column to input
    X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
    X_val = np.hstack((np.ones((X_val.shape[0], 1)), X_val))

    W1, W2, loss_history, acc_history = train_nn(
        X_train,
        y_train,
        X_val,      # pass validation fold
        y_val,
        lr=0.05,
        hidden_size=5,
        lambda_reg=0.001
    )

    val_accuracy = evaluate_nn(X_val, y_val, W1, W2)
    fold_accuracies.append(val_accuracy)

    print(f"Fold {fold_no}: Validation Accuracy = {val_accuracy:.4f}")


Fold 1: Validation Accuracy = 0.7273
Fold 2: Validation Accuracy = 0.7922
Fold 3: Validation Accuracy = 0.7143
Fold 4: Validation Accuracy = 0.8442
Fold 5: Validation Accuracy = 0.8312
Fold 6: Validation Accuracy = 0.6883
Fold 7: Validation Accuracy = 0.8442
Fold 8: Validation Accuracy = 0.7792
Fold 9: Validation Accuracy = 0.6842
Fold 10: Validation Accuracy = 0.7895


In [67]:
mean_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

print("\n10-Fold Cross Validation Results")
print("Fold Accuracies:", [f"{acc:.4f}" for acc in fold_accuracies])
print(f"Average Validation Accuracy: {mean_accuracy:.4f}")
print(f"Standard Deviation: {std_accuracy:.4f}")


10-Fold Cross Validation Results
Fold Accuracies: ['0.7273', '0.7922', '0.7143', '0.8442', '0.8312', '0.6883', '0.8442', '0.7792', '0.6842', '0.7895']
Average Validation Accuracy: 0.7694
Standard Deviation: 0.0589
